# Parallel LLM Decoding: Speculative and Diffusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/llm/parallel_token_generation_diffusion.ipynb)

Autoregressive LLMs emit one token per forward pass, the core latency wall of
inference. This notebook builds two ways to break it, from scratch in NumPy:

1. **Masked-diffusion decoding** generates a whole block in parallel and refines
   it over K denoising steps. We measure how parallelism trades against quality.
2. **Speculative decoding** lets a cheap draft propose and an expensive target
   verify in one parallel pass, staying provably exact while running faster.

We use a known toy "language" (an order-1 Markov chain) so we have exact
conditionals and can isolate the effect of *parallelism*, not model quality.
This is conceptual: the real systems run on LLMs and CUDA. We illustrate the
mechanisms, then explain how **Orthrus** (2026) composes them.

**Blog post:** [sesen.ai/blog/parallel-token-generation-diffusion-decoding](https://sesen.ai/blog/parallel-token-generation-diffusion-decoding)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

NAVY, TEAL, AMBER, GREY = "#1B2D3D", "#3D9B8F", "#D4A24C", "#C2CAD0"
MASK = -1
np.random.seed(0)

## Part 1: A toy "language"

Our ground truth is a low-entropy order-1 Markov chain over a six-token
vocabulary: each token strongly prefers a few successors, so sequences have real
structure. Because we know the transition matrix `T`, we can compute exact
conditionals later.

In [ ]:
def make_language(vocab=6, seed=0, concentration=0.4):
    rng = np.random.default_rng(seed)
    T = rng.dirichlet(np.full(vocab, concentration), size=vocab)   # rows sum to 1
    vals, vecs = np.linalg.eig(T.T)                                # stationary dist
    pi = np.real(vecs[:, np.argmin(np.abs(vals - 1))])
    return T, np.abs(pi) / np.abs(pi).sum()


def seq_logprob(seq, T, pi):
    lp = np.log(pi[seq[0]] + 1e-12)
    for a, b in zip(seq[:-1], seq[1:]):
        lp += np.log(T[a, b] + 1e-12)
    return lp


def ar_sample(T, pi, L, rng):                       # autoregressive reference
    seq = [rng.choice(len(pi), p=pi)]
    for _ in range(L - 1):
        seq.append(rng.choice(len(T), p=T[seq[-1]]))
    return np.array(seq)

T, pi = make_language()
print("transition matrix T (rows = current token, cols = next token):")
print(np.round(T, 2))

## Part 2: Masked-diffusion decoding

Start from an all-masked block and reveal a few tokens per step, each sampled from
its conditional given the neighbours already revealed. Tokens revealed in the same
step are sampled independently, so they cannot see each other.

In [ ]:
def denoiser_conditional(left, right, T, pi):
    # p(x_t | visible neighbours) under the true order-1 chain
    if left != MASK and right != MASK:
        p = T[left, :] * T[:, right]
    elif left != MASK:
        p = T[left, :].copy()
    elif right != MASK:
        p = pi * T[:, right]
    else:
        p = pi.copy()
    return p / p.sum()


def schedule(L, K):
    base = [L // K] * K
    for i in range(L - sum(base)):
        base[i] += 1
    return base


def diffusion_decode(T, pi, L, K, rng):
    seq = np.full(L, MASK)
    for n_reveal in schedule(L, K):
        masked = [t for t in range(L) if seq[t] == MASK]
        if not masked:
            break
        conds = {t: denoiser_conditional(seq[t-1] if t > 0 else MASK,
                                         seq[t+1] if t < L-1 else MASK, T, pi)
                 for t in masked}
        order = sorted(masked, key=lambda t: -conds[t].max())   # most confident first
        for t in order[:n_reveal]:
            seq[t] = rng.choice(len(pi), p=conds[t])
    return seq

rng = np.random.default_rng(7)
print("K=1  (fully parallel):", diffusion_decode(T, pi, 12, 1, rng))
print("K=12 (autoregressive):", diffusion_decode(T, pi, 12, 12, rng))

Now measure quality (negative log-likelihood under the true model) versus the number of denoising steps. One step is the worst; more steps recover the autoregressive floor.

In [ ]:
L, N = 16, 4000
rng = np.random.default_rng(1)
ar = np.mean([-seq_logprob(ar_sample(T, pi, L, rng), T, pi) / L for _ in range(N)])
Ks = [1, 2, 4, 8, 16]
nll = [np.mean([-seq_logprob(diffusion_decode(T, pi, L, K, rng), T, pi) / L
                for _ in range(N)]) for K in Ks]

plt.figure(figsize=(8, 5))
plt.xscale("log", base=2)
plt.plot(Ks, nll, "o-", color=TEAL, lw=2.5, ms=8, label="masked diffusion")
plt.axhline(ar, ls="--", color=NAVY, lw=2, label=f"autoregressive floor ({ar:.2f})")
plt.xticks(Ks, Ks); plt.xlabel("denoising steps K (log scale)")
plt.ylabel("NLL / token (lower = better)"); plt.legend(frameon=False); plt.grid(alpha=0.2, which="both")
plt.title("More steps, more quality, less parallelism"); plt.show()
print("K=1 NLL:", round(nll[0], 3), " AR floor:", round(ar, 3))

**The independence problem.** At `K=1`, every token is sampled from its marginal
with no coordination, so dependencies are broken (individually plausible tokens,
incoherent sequence). Each extra step lets later tokens condition on earlier ones,
rebuilding the joint distribution. `K=L` recovers autoregressive quality exactly.
Denoising steps are a dial between speed and quality.

## Part 3: Speculative decoding (parallel, but exact)

A cheap **draft** proposes the next gamma tokens; the expensive **target** verifies
them all in one parallel pass. An accept-or-resample test keeps the output exactly
target-distributed, no matter how bad the draft is.

In [ ]:
def make_draft(T, eps):                 # target blended toward uniform
    V = T.shape[0]
    return (1 - eps) * T + eps * np.full((V, V), 1.0 / V)


def speculative_decode(T, Td, pi, n_tokens, gamma, rng):
    seq = [rng.choice(len(pi), p=pi)]
    target_passes = 0
    while len(seq) < n_tokens + 1:
        ctx = seq[-1]
        proposed, q = [], []; c = ctx
        for _ in range(gamma):                       # draft proposes gamma tokens
            x = rng.choice(len(Td[c]), p=Td[c]); proposed.append(x); q.append(Td[c]); c = x
        target_passes += 1                           # target verifies in ONE pass
        c = ctx
        for i, x in enumerate(proposed):
            p = T[c]
            if rng.random() <= min(1.0, p[x] / (q[i][x] + 1e-12)):
                seq.append(x); c = x                 # accept
            else:
                adj = np.maximum(p - q[i], 0)         # reject: resample, stop
                seq.append(rng.choice(len(p), p=adj / adj.sum())); break
        else:
            seq.append(rng.choice(len(T), p=T[c]))   # all accepted: bonus token
    return np.array(seq[:n_tokens + 1]), target_passes

Check exactness: compare the transition statistics of speculative output against direct target sampling. They should be identical (KL near 0).

In [ ]:
def empirical_transition(seq, V):
    M = np.zeros((V, V)) + 1e-9
    for a, b in zip(seq[:-1], seq[1:]):
        M[a, b] += 1
    return M / M.sum(1, keepdims=True)


def kl_rows(P, Q):
    return float(np.mean(np.sum(P * np.log((P + 1e-12) / (Q + 1e-12)), axis=1)))

V = len(pi); rng = np.random.default_rng(2)
spec, _ = speculative_decode(T, make_draft(T, 0.4), pi, 120000, 4, rng)
direct = [rng.choice(V, p=pi)]
for _ in range(120000):
    direct.append(rng.choice(V, p=T[direct[-1]]))
Pspec, Pdir = empirical_transition(spec, V), empirical_transition(np.array(direct), V)
kl = kl_rows(Pdir, Pspec)

plt.figure(figsize=(5.5, 5.5))
plt.plot([0, 1], [0, 1], ls="--", color=GREY)
plt.scatter(Pdir.ravel(), Pspec.ravel(), s=50, color=TEAL, edgecolors="white")
plt.xlabel("direct target sampling"); plt.ylabel("speculative decoding")
plt.title(f"Speculative decoding is exact (KL = {kl:.4f})"); plt.show()
print("KL(direct || speculative) =", round(kl, 5))

Now the speedup: tokens generated per expensive target pass, as a function of the draft's acceptance rate and the block size gamma.

In [ ]:
plt.figure(figsize=(8, 5))
for gamma, col in [(2, AMBER), (4, TEAL), (8, NAVY)]:
    acc, tpp = [], []
    for eps in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
        rng = np.random.default_rng(3)
        seq, passes = speculative_decode(T, make_draft(T, eps), pi, 30000, gamma, rng)
        t = (len(seq) - 1) / passes
        tpp.append(t); acc.append((t - 1) / gamma)
    plt.plot(acc, tpp, "o-", color=col, lw=2.5, label=f"gamma = {gamma}")
plt.axhline(1.0, ls="--", color=GREY, label="autoregressive (1/pass)")
plt.xlabel("draft acceptance rate"); plt.ylabel("tokens per target pass (speedup)")
plt.legend(frameon=False); plt.grid(alpha=0.2)
plt.title("Speculative speedup grows with draft acceptance"); plt.show()

## Part 4: The draft's memory cost, and Orthrus

A separate draft model carries its own KV cache, which grows with context length
just like the target's. At long contexts you pay for two growing caches.

In [ ]:
n = np.linspace(0, 32000, 200)
target = 1.0 * n
draft_based = (1.0 + 0.25) * n                 # target + separate draft cache
shared = 1.0 * n + 0.02 * n.max()              # one shared cache + O(1) head
plt.figure(figsize=(8, 5))
plt.plot(n/1000, draft_based, color=AMBER, lw=2.5, label="separate draft cache (EAGLE-style)")
plt.plot(n/1000, shared, color=TEAL, lw=2.5, label="shared cache (Orthrus)")
plt.plot(n/1000, target, color=GREY, ls=":", lw=1.5, label="plain autoregressive")
plt.xlabel("context length (thousands of tokens)"); plt.ylabel("KV-cache memory (relative)")
plt.legend(frameon=False); plt.grid(alpha=0.2)
plt.title("Shared cache vs separate draft cache (illustrative)"); plt.show()

**Orthrus (2026)** removes the draft's separate cache. It attaches a lightweight,
trainable *diffusion view* to a *frozen* autoregressive model; both views attend to
the **same KV cache**. The diffusion view proposes a block in parallel, the
autoregressive view verifies it with the same accept-or-resample logic we built
above, so the output is provably identical to the frozen model's, a **lossless**
speedup of up to 7.8x. Only ~16% of parameters are trained (LoRA-style), and the
shared cache keeps the speedup as context grows, where draft-model methods lose
ground to their own caches. See the [official implementation](https://github.com/chiennv2000/orthrus).

It is the two halves of this notebook combined: diffusion's parallel block proposal,
wrapped in speculative decoding's exact verification, sharing one cache.

## Exercises

1. **Higher-order language.** Make the toy language an order-2 Markov chain and
   update `denoiser_conditional`. Does diffusion need more steps to recover quality?
2. **Block speculative decoding.** Combine the two: use `diffusion_decode` as the
   draft proposer and verify the block with the speculative accept/reject test.
   Confirm the output stays exact and measure the speedup.
3. **Acceptance vs gamma.** For a fixed draft quality, find the gamma that maximises
   tokens-per-pass. Why does very large gamma stop helping?
4. **Adaptive steps.** In `diffusion_decode`, stop early once all conditionals are
   confident (max prob above a threshold). How few steps can you use without losing
   quality?
5. **Real models.** Install `vllm` and enable speculative decoding with a small draft
   model; compare its tokens/sec against vanilla decoding on your hardware.